# Homework 4: Optional Advanced Final Project (A+ Grade Bump)

## Parliamentary Attention and Stance toward Skilled Immigration in Japan, 2012–2024

- **Student Name:** Cao Han|37255239
- **Policy relevance:** Japan increasingly depends on foreign workers while immigration remains politically contested. Measuring parliamentary attention, stance, and speaker concentration helps distinguish broad policy concern from rhetoric driven by a small number of legislators.
- **Central hypothesis:** Immigration-related parliamentary attention increased over the study period, but the pattern is event-driven rather than a smooth linear rise.
- **Scope:** Japanese National Diet speeches from 2012–2024, using audited aggregate outputs from the completed Skilled-Immigration project.

### Why this is the advanced version

This notebook uses an object-oriented, configuration-driven pipeline. Paths, filenames, visualization settings, and model variables are stored in `project_settings.yaml`. The workflow is divided into four reusable classes:

1. `DataAcquisition`
2. `DataManipulation`
3. `DataVisualization`
4. `EconometricModeling`

The large 351 MB raw-text corpus and multi-hour LLM classification stage are not committed because of the course file-size limit. Instead, compact aggregate snapshots from the completed pipeline are committed so every cell below runs out of the box.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the course repository.")


PROJECT_ROOT = find_project_root()
ADVANCED_DIR = PROJECT_ROOT / "src" / "final_project" / "skilled_immigration" / "advanced"
CONFIG_PATH = PROJECT_ROOT / "references" / "configs" / "final_project" / "skilled_immigration" / "project_settings.yaml"
if str(ADVANCED_DIR) not in sys.path:
    sys.path.insert(0, str(ADVANCED_DIR))

print("Project root located successfully.")
print(f"Configuration: {CONFIG_PATH.relative_to(PROJECT_ROOT)}")

Project root located successfully.
Configuration: references/configs/final_project/skilled_immigration/project_settings.yaml


## 1. Acquire Data with `DataAcquisition`

The source project collected 10,961 speeches from the National Diet Library Diet Minutes API and classified immigration-related text windows as `pro`, `neutral`, or `anti`. The advanced acquisition class reads four committed aggregate snapshots and validates their schemas before returning them as a dictionary of DataFrames.

In [2]:
from data import DataAcquisition

data_fetcher = DataAcquisition(config_path=CONFIG_PATH, project_root=PROJECT_ROOT)
raw = data_fetcher.run()
print({name: frame.shape for name, frame in raw.items()})
print(raw["annual"].head().to_string(index=False))

{'annual': (13, 2), 'stance': (3, 2), 'concentration': (2, 5), 'correlation': (1, 4)}
 year  n_speeches
 2012          52
 2013         112
 2014         589
 2015         347
 2016        1470


### Data acquisition details

- **Primary source:** National Diet Library Diet Minutes API
- **Study period:** 2012–2024
- **Keywords:** 技能実習, 外国人労働者, 出入国管理, 特定技能, 外国人材, 不法滞在, 多文化共生, 移民政策, 外国人犯罪
- **Pipeline scale:** 10,961 speeches and approximately 20,741 extracted text windows
- **Committed inputs:** annual counts, stance counts, concentration statistics, and the prefecture-level correlation result

## 2. Manipulate Data with `DataManipulation`

The manipulation class converts data types, removes duplicate years, orders stance labels, calculates stance shares, creates a centered time variable, and adds a three-year rolling mean for visualization. Cleaned outputs are saved automatically under the configured processed-data directory.

In [3]:
from manipulate import DataManipulation

manipulator = DataManipulation(config_path=CONFIG_PATH, project_root=PROJECT_ROOT)
clean = manipulator.run()
print(clean["annual"].to_string(index=False))

 year  n_speeches  year_centered  rolling_mean  year_over_year_change
 2012          52              0     52.000000                    NaN
 2013         112              1     82.000000             115.384615
 2014         589              2    251.000000             425.892857
 2015         347              3    349.333333             -41.086587
 2016        1470              4    802.000000             323.631124
 2017         485              5    767.333333             -67.006803
 2018        2573              6   1509.333333             430.515464
 2019        1699              7   1585.666667             -33.968131
 2020         494              8   1588.666667             -70.924073
 2021         396              9    863.000000             -19.838057
 2022         470             10    453.333333              18.686869
 2023         707             11    524.333333              50.425532
 2024        1560             12    912.333333             120.650636


In [4]:
print(clean["stance"].to_string(index=False))

 stance  n_speeches  share_pct
     pro        1497  14.137312
 neutral        6905  65.209179
    anti        2187  20.653508


### Preprocessing summary

- The annual series contains 13 observations from 2012 through 2024.
- `year_centered` equals zero in 2012 and is used as the independent variable in the trend model.
- A configurable three-year rolling average is generated for the advanced time-series visualization.
- Neutral speeches account for about 65.2%, anti speeches for 20.7%, and pro speeches for 14.1%.

## 3. Visualize Data with `DataVisualization`

The visualization class reads its output directory and DPI from the YAML configuration. It produces three publication-ready figures with titles, axis labels, and legends.

In [5]:
from graph import DataVisualization

visualizer = DataVisualization(config_path=CONFIG_PATH, project_root=PROJECT_ROOT)
figure_paths = visualizer.run()
for figure_path in figure_paths:
    print(figure_path.relative_to(PROJECT_ROOT))

notebooks/hw/final_project/figures/advanced/advanced_attention_trend.svg
notebooks/hw/final_project/figures/advanced/advanced_stance_distribution.svg
notebooks/hw/final_project/figures/advanced/advanced_speaker_concentration.svg


### Rendered figures

#### Parliamentary attention and three-year rolling mean

![Advanced attention trend](figures/advanced/advanced_attention_trend.svg)

#### Overall stance distribution

![Advanced stance distribution](figures/advanced/advanced_stance_distribution.svg)

#### Concentration of anti-immigration speech

![Advanced speaker concentration](figures/advanced/advanced_speaker_concentration.svg)

### Visualization findings

- **Attention trend:** Speech volume is highly episodic. The strongest peak occurs in 2018, followed by elevated activity in 2019 and 2024. The rolling mean makes the broader period of heightened attention visible without hiding year-specific shocks.
- **Stance distribution:** Most classified speeches are neutral. Anti-immigration speeches are more common than pro-immigration speeches, but neither category is the majority.
- **Speaker concentration:** Nationally, the top anti-immigration speaker accounts for only 6.2% of such speeches. Within the median prefecture, however, the top speaker accounts for 51.9% and the top three account for 94.7%, showing that local rhetoric is often dominated by a few legislators.

## 4. Model Data with `EconometricModeling`

The configured OLS model is:

`annual speeches_t = beta_0 + beta_1 × (year_t - 2012) + error_t`

The dependent and independent variable names are read from `project_settings.yaml`, allowing the model class to be reused with different specifications.

In [6]:
from model import EconometricModeling

modeler = EconometricModeling(config_path=CONFIG_PATH, project_root=PROJECT_ROOT)
result = modeler.run()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:             n_speeches   R-squared:                       0.098
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     1.199
Date:                Thu, 16 Jul 2026   Prob (F-statistic):              0.297
Time:                        06:58:47   Log-Likelihood:                -103.28
No. Observations:                  13   AIC:                             210.6
Df Residuals:                      11   BIC:                             211.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           481.4286    388.756      1.238

In [7]:
import pandas as pd

coefficient_table = pd.DataFrame({
    "coefficient": result.params,
    "std_error": result.bse,
    "p_value": result.pvalues,
})
print(coefficient_table.to_string())

               coefficient   std_error   p_value
const           481.428571  388.756116  0.241352
year_centered    60.197802   54.978417  0.296940


### Model interpretation

The estimated trend is approximately **+60 speeches per year**, but it is not statistically significant at conventional levels (**p ≈ 0.297, R² ≈ 0.098, n = 13**). The central hypothesis is therefore only partly supported: later years sometimes have higher attention, but a simple linear trend does not explain the large event-driven fluctuations.

The completed project's supplementary prefecture-level analysis also found virtually no relationship between foreign-resident share and anti-immigration speech share (**Pearson r = −0.050, p = 0.7489, n = 46**). Geographic exposure alone therefore does not explain parliamentary rhetoric.

## 5. Run the Complete OOP Pipeline

The orchestration class runs all four stages with one command. From the repository root:

```bash
uv run python src/final_project/skilled_immigration/advanced/pipeline.py
```

In [8]:
from pipeline import SkilledImmigrationPipeline

pipeline = SkilledImmigrationPipeline(config_path=CONFIG_PATH, project_root=PROJECT_ROOT)
outputs = pipeline.run()
summary = {
    "raw_datasets": list(outputs["raw"]),
    "clean_datasets": list(outputs["clean"]),
    "figures_generated": len(outputs["figures"]),
    "regression_observations": int(outputs["regression"].nobs),
}
print(summary)

{'raw_datasets': ['annual', 'stance', 'concentration', 'correlation'], 'clean_datasets': ['annual', 'stance', 'concentration', 'correlation'], 'figures_generated': 3, 'regression_observations': 13}


## Conclusion

The advanced pipeline confirms that immigration-related attention in the Japanese Diet is shaped by policy events rather than a stable linear increase. Most speeches are neutral, the prefecture-level exposure correlation is essentially zero, and local anti-immigration discourse is frequently concentrated among very few legislators. The OOP design separates data loading, transformation, visualization, and modeling, while the configuration file makes paths and analysis settings easy to change without rewriting the classes.